# McDonald's atsiliepimų sentimentų analizė

Šiame užrašų bloknote parodomi pagrindiniai sentimentų analizės rezultatai ir modelių palyginimas, naudojant McDonald's klientų atsiliepimų duomenis.


In [ ]:
# Importuojamos reikalingos funkcijos ir bibliotekos
import sys
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from train_models import (
    ensure_utf8_stdout,
    load_and_clean_data,
    vectorize_text,
    train_models,
    evaluate_models,
    add_needs_urgent_response_column,
)

ensure_utf8_stdout()

# 1. Duomenų įkėlimas ir papildomas valymas
df = load_and_clean_data("McDonald_s_Reviews_prepared.csv")
print(f"Eilučių skaičius po valymo: {len(df)}")

# 2. Sentimentų pasiskirstymas
plt.figure(figsize=(4, 3))
sns.countplot(x="sentiment", data=df)
plt.title("Sentimentų pasiskirstymas (0 = neigiamas, 1 = teigiamas)")
plt.tight_layout()
plt.show()

# 3. TF-IDF vektorizacija ir duomenų skaidymas
from sklearn.model_selection import train_test_split

X, y, vectorizer = vectorize_text(df)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 4. Modelių mokymas ir vertinimas
models = train_models(X_train, y_train)
results_df = evaluate_models(models, X_train, X_test, y_train, y_test)
results_df


In [ ]:
# 5. Verslo logika: `needs_urgent_response` ir keli pavyzdžiai
best_model_name = results_df.iloc[0]["model"]
best_model = models[best_model_name]
print(f"Geriausias modelis pagal F1-score: {best_model_name}")

df_with_flags = add_needs_urgent_response_column(df, best_model, vectorizer)
print("Pirmos 5 eilutės su `needs_urgent_response`:")
df_with_flags[["review", "sentiment", "needs_urgent_response"]].head()
